<a href="https://colab.research.google.com/github/Rrricardi/Quantum-Machine-Learning-QML-y-Simulaci-n-F-sica/blob/main/Quantum%20Machine%20Learning%20(QML)%20y%20Simulaci%C3%B3n%20F%C3%ADsica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pennylane
import pennylane as qml
from pennylane import numpy as pnp
import torch
import torch.nn as nn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

# 1. Preparación y Reducción de Datos
n_features = 4  # Determina el número de cúbits
n_qubits = n_features
n_layers = 3

X_raw, y_raw = make_classification(
    n_samples=300,
    n_features=8,
    n_informative=4,
    n_classes=2,
    random_state=42
)

# Reducción PCA a 4 componentes y escalado a [0, pi]
pca = PCA(n_components=n_features)
X_pca = pca.fit_transform(X_raw)
scaler = MinMaxScaler(feature_range=(0, torch.pi))
X_scaled = scaler.fit_transform(X_pca)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_raw, test_size=0.2, random_state=42
)

# Conversión a tensores de PyTorch
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# 2. Definición del Circuito Cuántico (QNode)
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="parameter-shift")
def quantum_circuit(inputs, weights):
    # Codificación de datos
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")

    # Capas variacionales entrelazadas
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))

    # Medición del observable Pauli Z en el cúbit 0
    return qml.expval(qml.PauliZ(0))

# 3. Integración en Módulo Híbrido PyTorch
class HybridVQC(nn.Module):
    def __init__(self, n_qubits, n_layers):
        super().__init__()
        weight_shapes = {"weights": (n_layers, n_qubits, 3)}
        self.qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)
        self.post_process = nn.Sequential(
            nn.Linear(1, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # La salida cuántica está en [-1, 1], se mapea a [0, 1]
        q_out = self.qlayer(x)
        if q_out.ndim == 1:
            q_out = q_out.unsqueeze(1)
        return self.post_process(q_out)

# 4. Entrenamiento y Optimización
model = HybridVQC(n_qubits, n_layers)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)

epochs = 30
batch_size = 16

for epoch in range(epochs):
    permutation = torch.randperm(X_train_t.size()[0])
    epoch_loss = 0.0

    for i in range(0, X_train_t.size()[0], batch_size):
        indices = permutation[i:i + batch_size]
        batch_x, batch_y = X_train_t[indices], y_train_t[indices]

        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            preds_test = (model(X_test_t) >= 0.5).float()
            acc = (preds_test == y_test_t).float().mean()
            print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} - Test Acc: {acc:.2%}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 38.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 35.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 56.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 79.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 58.8 MB/s  0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [pennylane]
Epoch [10/30] - Loss: 6.8594 - Test Acc: 88.33%
Epoch [20/30] - Loss: 5.2170 - Test Acc: 88.33%
Epoch [30/30] - Loss: 5.0186 - Test Acc: 88.33%
